# Data Exchange via McK Analytics S3 service

This notebook demonstrates the flow for securely processing client data using the McK Analytics S3 service.
It covers reading client data from S3, processing it by using MLRun (see example below), and uploading the results back to McK Analytics buckets for client to download. For more information see [Knowledge-base](https://platform.mckinsey.com/knowledge-base/service-guide/KO89357/uploading-data-to-s3?searchTerm=s3).

## Prerequisite

1. Create an .env file
With those envs and the creds from the AWS S3 service.

```
AWS_ACCESS_KEY_ID_MCK=...
AWS_SECRET_ACCESS_KEY_MCK=...
S3_BUCKET_MCK=...
```
2. The client uploaded the file using ``toMcK`` link.

3. Once the user uploads the file to the ``toMcK`` folder it will automatically forward it to this path:
``s3://<McK-Analytics-Bucket>/Raw/incoming/``.

4. Move the file to the ``Master`` directory:
``s3://<McK-Analytics-Bucket>/Master/``.


6. See below an example how to acces and modify the uploaded file by using MLRun.

**Note** - The credetials allow to read and write only from the ``Master`` directory.


## Step 1 - Import libraries, load environment variables and create a project
Set up the required Python libraries and load AWS credentials from an environment file for secure access to the client S3 bucket.

In [1]:
import mlrun
import dotenv  # load the env file for the client s3 bucket
import os

dotenv.load_dotenv()

True

In [2]:
project = mlrun.get_or_create_project("s3-mck")

> 2025-12-16 09:34:06,208 [info] Project loaded successfully: {"project_name":"s3-mck"}


## Step 2 - Configure and register the McK Analytics S3 as datastore profile

Define an S3 datastore profile using the client credentials and register them as a ``DatastoreProfileS3``, by using datastore profile you can access the S3 bucket with the AWS credentials.

See also - [S3 data store profile](https://docs.mlrun.org/en/stable/store/datastore.html#s3)

In [3]:
from mlrun.datastore.datastore_profile import DatastoreProfileS3

s3_profile = DatastoreProfileS3(
    name="s3-mck",
    access_key_id=os.environ["AWS_ACCESS_KEY_ID_MCK"],
    secret_key=os.environ["AWS_SECRET_ACCESS_KEY_MCK"],
    bucket=os.environ["S3_BUCKET_MCK"],
)
project.register_datastore_profile(s3_profile)
project.set_secrets({"mck_s3_path": f"ds://{s3_profile.name}/Master"})

## Step 3 - Define the processing function

Create a Python function that reads the client data from S3, processes it by adding a new column, and prepares it for logging.

In MLRun, the artifact path allows you to explicitly define the destination where files or datasets will be uploaded and stored.

For example:
* For Datastore profile with the name s3 that uses a bucket name: test
* For artifact path - ``ds://s3/Master/``
* file.txt - will be stored in ``s3://test/Master``

In [4]:
%%writefile func.py
import mlrun
import os

def func(context: mlrun.MLClientCtx,file_name: str):
    di = mlrun.get_dataitem(f'{os.environ["mck_s3_path"]}/{file_name}') #Get the data from the MCK S3 bucket
    df = di.as_df()
    context.logger.info(f"Client Data: {df.values.tolist()}")
    df["new_data"]=[4]*df.shape[0] # Create a new column of data 
    context.log_dataset("new_data_mlrun",df=df) # Log the dataset to the s3 instance bucket
    context.log_dataset("new_data_mck",df=df,artifact_path=os.environ["mck_s3_path"]) # Log the dataset to the s3 analytics bucket

    return 1

Overwriting func.py


## Step 4 - Set & run MLRun function 

Register the function file as an MLRun job, specifying the runtime image and handler.

In [5]:
func = project.set_function(
    func="func.py", name="func", handler="func", image="mlrun/mlrun", kind="job"
)

In [6]:
# This example uses this file "sample_data.csv", please change to your file name
func.run(params={"file_name": "sample_data.csv"})

> 2025-12-16 09:34:06,337 [info] Storing function: {"db":"http://mlrun-api:8080","name":"func-func","uid":"9685f7a4972840e9b663cecd616eb5ce"}
> 2025-12-16 09:34:06,627 [info] Job is running in the background, pod: func-func-v49xb
> 2025-12-16 09:34:20,606 [info] Client Data: [[1, 'Alice', 25, 'TelAviv', 88], [2, 'Bob', 30, 'Haifa', 92], [3, 'Carol', 27, 'Jerusalem', 85], [4, 'David', 35, 'RamatGan', 90], [5, 'Eva', 22, 'Herzliya', 78], [6, 'Frank', 40, 'Netanya', 95], [7, 'Grace', 29, 'BeerSheva', 83], [8, 'Helen', 33, 'Holon', 89], [9, 'Ian', 26, 'BatYam', 80], [10, 'Judy', 31, 'PetahTikva', 91]]
> 2025-12-16 09:34:22,004 [info] To track results use the CLI: {"info_cmd":"mlrun get run 9685f7a4972840e9b663cecd616eb5ce -p s3-mck","logs_cmd":"mlrun logs 9685f7a4972840e9b663cecd616eb5ce -p s3-mck"}
> 2025-12-16 09:34:22,005 [info] Run execution finished: {"name":"func-func","status":"completed"}


project,uid,iter,start,end,state,kind,name,labels,inputs,parameters,results,artifacts
s3-mck,...616eb5ce,0,Dec 16 09:34:19,2025-12-16 09:34:21.988948+00:00,completed,run,func-func,kind=jobowner=jovyanmlrun/client_version=1.9.2mlrun/client_python_version=3.11.13host=func-func-v49xb,,file_name=sample_data.csv,return=1,new_data_mcknew_data_mlrun


> 2025-12-16 09:34:26,747 [info] Run execution finished: {"name":"func-func","status":"completed"}


## Step 5 - Access McK Analytics S3 & download modified file   
After completing the steps above, the file will be stored in ``s3://<McK-Analytics-Bucket>/Master/``.
Move the file from ``Master`` directory to the ``fromMcK`` for client to access the modified file.